In [ ]:
## requires 
# pip install snowflake-connector-python[pandas]
# pip install sqlalchemy

Run Cells Sequentially
Each cell is numbered in the order they should be run in

In [22]:
#imports
import snowflake.connector
import pandas as pd
from sqlalchemy import VARCHAR, false, table
import os
import json
from configparser import ConfigParser,DuplicateSectionError
import re

In [23]:
#utilities
def df(conn,sql):
    connCur = conn.cursor()
    connCur.execute(sql)
    df = pd.DataFrame(connCur.fetch_pandas_all())
    connCur.close()
    return df

def listJson(conn,sql):
    connCur = conn.cursor()
    connCur.execute(sql)
    df = connCur.fetchall()
    connCur.close()
    return df

def getDDL(conn,sql):
    connCur = conn.cursor()
    result = connCur.execute(sql).fetchone()
    connCur.close()
    return result

def getFileList(path):
    # get file list for path
    filesList = []
    for x in os.listdir(path):
        z = x.upper()
        if z.endswith('.CSV') and z.startswith('ZCDV'):
            filesList.append(path + "\\" + x)
    return filesList

def getFile(path):
    f = open(path,'r')
    filein = f.readlines()
    f.close()
    return filein

def writeFile(path,fileOut):
    f = open(path,'w')
    f.write(fileOut)
    f.close()

def writeConfigParser(path,parserFile):
    f = open(path, 'w') 
    parserFile.write(f)

def getLastSnow(snowpath, kanbanid):
    current_files = os.listdir(snowpath)
    matches = re.findall(r'K\.'+kanbanid+'\.(\d+)__.*?\.sql', str(current_files))
    if matches:
        return int(max(matches))
    return 0

In [24]:
#set vars
lzSchema = "SAPAPI"
odsSchema = "SAPAPI"
overwriteDDLs = True
#snow file paths, please make sure you update this to the correct file name
snowLZ_path = "C:\\Repo\\EnterpriseCloud\\Snow\\K.28642.01__sapapiLZTables.sql" 
snowODS_path = "C:\\Repo\\EnterpriseCloud\\Snow\\K.28642.02__Create_sapapi_ODSTables.sql"
configFile = "C:\\Repo\\EnterpriseCloud\\Configuration\\FileUploader_SAPAPI.json"
apiFiles = "C:\\Users\\221291\\Documents\\" 
cdsViewConfigFile = "C:\\Repo\\EnterpriseCloud\\Configuration\\SAP_cdsv.ini"
repo_path = "C:\\Repo\\EnterpriseCloud\\"
templatePath = repo_path + "Notebooks\\CSVTemplates"
refTextPath = repo_path + "Notebooks"

typeDict = {'CHAR':'VARCHAR(16777216)',
            'CLNT':'VARCHAR(16777216)',
            'DATS':'VARCHAR(16777216)',
            'TIMS':'VARCHAR(16777216)',
            'CUKY':'VARCHAR(16777216)',
            'CURR':'VARCHAR(16777216)',
            'DEC':'VARCHAR(16777216)',
            'NUMC':'VARCHAR(16777216)',
            'QUAN':'VARCHAR(16777216)',
            'UNIT':'VARCHAR(16777216)',
            'FLTP':'VARCHAR(16777216)',
            'INT4':'VARCHAR(16777216)',
            'INT2':'VARCHAR(16777216)',
            'INT1':'VARCHAR(16777216)',
            'RAW':'VARCHAR(16777216)',
            'LANG':'VARCHAR(16777216)',
            'ACCP':'VARCHAR(16777216)',
            'SSTRING':'VARCHAR(16777216)'}

In [25]:
# instantiate file list,  df_tables dataframe, and sf_characters
filesList = getFileList(templatePath)

df_tables = pd.DataFrame()

# read csv template
for file in filesList:
    csvData = pd.read_csv(file,nrows=1)
    #print(csvData)    
    df_tables = df_tables.append(csvData,ignore_index=True)

df_tables["TABLE_NAME"] = df_tables["CDS View"] + "_" +  df_tables["Source.Name"]
#print(df_tables)

df_characters = pd.read_csv(refTextPath + "\\SF_characters.csv")
#print(df_characters)

In [26]:
# create LZ tables ddl

if overwriteDDLs:
    ddlAction = """create or replace TABLE """ + df_tables['TABLE_NAME']
else:
    ddlAction = """create TABLE """ + df_tables['TABLE_NAME'] + """ if not exists"""
    
#generate DDL
df_tables['lz_ddl'] = ddlAction + """ (\
_FILENAME VARCHAR(16777216),\
_FILEDATE TIMESTAMP_NTZ(9),\
_UPLOADDATE TIMESTAMP_NTZ(9),\
JSONFIELD VARIANT\
)COMMENT='Model SAPAPI """ + df_tables['TABLE_NAME'] + """ file.';"""

print(df_tables['lz_ddl'].values)
    

["create or replace TABLE ZCDV_T_CMTU062_T001 (_FILENAME VARCHAR(16777216),_FILEDATE TIMESTAMP_NTZ(9),_UPLOADDATE TIMESTAMP_NTZ(9),JSONFIELD VARIANT)COMMENT='Model SAPAPI ZCDV_T_CMTU062_T001 file.';"
 "create or replace TABLE ZCDV_T_CMTU087_ADRC (_FILENAME VARCHAR(16777216),_FILEDATE TIMESTAMP_NTZ(9),_UPLOADDATE TIMESTAMP_NTZ(9),JSONFIELD VARIANT)COMMENT='Model SAPAPI ZCDV_T_CMTU087_ADRC file.';"
 "create or replace TABLE ZCDV_T_CMTU092_CSKT (_FILENAME VARCHAR(16777216),_FILEDATE TIMESTAMP_NTZ(9),_UPLOADDATE TIMESTAMP_NTZ(9),JSONFIELD VARIANT)COMMENT='Model SAPAPI ZCDV_T_CMTU092_CSKT file.';"
 "create or replace TABLE ZCDV_T_CMTU096_KNMT (_FILENAME VARCHAR(16777216),_FILEDATE TIMESTAMP_NTZ(9),_UPLOADDATE TIMESTAMP_NTZ(9),JSONFIELD VARIANT)COMMENT='Model SAPAPI ZCDV_T_CMTU096_KNMT file.';"
 "create or replace TABLE ZCDV_T_CMTU098_LFB1 (_FILENAME VARCHAR(16777216),_FILEDATE TIMESTAMP_NTZ(9),_UPLOADDATE TIMESTAMP_NTZ(9),JSONFIELD VARIANT)COMMENT='Model SAPAPI ZCDV_T_CMTU098_LFB1 file.';"


In [27]:
#write out LZ database table ddl commands to snow file
outstr = ""
outstr = "USE DATABASE <<LZ>>;\nUSE SCHEMA " + lzSchema + ";\n\n"
for idx, row in df_tables.iterrows():
    outstr += row['lz_ddl'] + """
"""
print(outstr)
writeFile(snowLZ_path,outstr)

USE DATABASE <<LZ>>;
USE SCHEMA SAPAPI;

create or replace TABLE ZCDV_T_CMTU062_T001 (_FILENAME VARCHAR(16777216),_FILEDATE TIMESTAMP_NTZ(9),_UPLOADDATE TIMESTAMP_NTZ(9),JSONFIELD VARIANT)COMMENT='Model SAPAPI ZCDV_T_CMTU062_T001 file.';
create or replace TABLE ZCDV_T_CMTU087_ADRC (_FILENAME VARCHAR(16777216),_FILEDATE TIMESTAMP_NTZ(9),_UPLOADDATE TIMESTAMP_NTZ(9),JSONFIELD VARIANT)COMMENT='Model SAPAPI ZCDV_T_CMTU087_ADRC file.';
create or replace TABLE ZCDV_T_CMTU092_CSKT (_FILENAME VARCHAR(16777216),_FILEDATE TIMESTAMP_NTZ(9),_UPLOADDATE TIMESTAMP_NTZ(9),JSONFIELD VARIANT)COMMENT='Model SAPAPI ZCDV_T_CMTU092_CSKT file.';
create or replace TABLE ZCDV_T_CMTU096_KNMT (_FILENAME VARCHAR(16777216),_FILEDATE TIMESTAMP_NTZ(9),_UPLOADDATE TIMESTAMP_NTZ(9),JSONFIELD VARIANT)COMMENT='Model SAPAPI ZCDV_T_CMTU096_KNMT file.';
create or replace TABLE ZCDV_T_CMTU098_LFB1 (_FILENAME VARCHAR(16777216),_FILEDATE TIMESTAMP_NTZ(9),_UPLOADDATE TIMESTAMP_NTZ(9),JSONFIELD VARIANT)COMMENT='Model SAPAPI ZC

In [28]:
# create ODS tables from standard CDS view templates to ODS snow file

# read csv template
outstr = "USE DATABASE <<ODS>>;\nUSE SCHEMA " + odsSchema + ";\n\n"
#print(filesList)
for file in filesList:
    columnList = ''
    csvData = pd.read_csv(file)
    #print(csvData.head(3))
    tableName = csvData.at[0,"CDS View"] + "_" + csvData.at[0,"Source.Name"]
    csvData = csvData.replace({'Type':typeDict})
   
    for idx,row in df_characters.iterrows():
        csvData["ScrubbedColumnName"]  = csvData["Technical Field Name(ODS)"].str.replace(row['OLD'],row['NEW'])
    csvData["ScrubbedColumnName"] = csvData["ScrubbedColumnName"].str.upper()
    csvData["columnList"] = '"' + csvData["ScrubbedColumnName"] + '" ' + csvData["Type"] + ","
    
    for idx,row in csvData.iterrows():
        columnList += row["columnList"] + '\n\t'
    if overwriteDDLs:
        ddlAction = 'create or replace TABLE ' + tableName + ' (\n\t'
    else:
        ddlAction = 'create TABLE ' + tableName  + ' if not exists (\n\t'
    controlColumns = '_FILEDATE TIMESTAMP_NTZ(9),\n\t_FILENAME varchar(128),\n\tAUDIT_LOG_ID varchar(128),\n\t_ROW_HASH varchar(128)\n);\n\n'
    
    outstr +=  ddlAction + columnList + controlColumns
    #print(outstr)
    #break

#print(ddlString) ## too big to print, check in file
writeFile(snowODS_path,outstr)


In [29]:
#Create cdsv config for SAPAPI code to use
filesList = getFileList(templatePath)
print(filesList)
#create configParser object
config = ConfigParser()
# get current config file and load into config
config.read(cdsViewConfigFile)

#creating config for files?

#SAP client for API request
SAPAPI_Client = 'client224'
#API user name
SAPAPI_user = 'IF_API_SNOWF'

# read csv template file and build list of required inputs
for csvFile in filesList:
    print(csvFile)
    dashChar = csvFile.rfind("-")
    dotChar = csvFile.rfind(".")
    dbt_strategy = csvFile[dashChar+1:dotChar]
    columnList = ""
    csvDataOut = pd.DataFrame()
    csvData = pd.read_csv(csvFile)
    print(csvData)
    cdsViewName = csvData.at[0,"CDS View"]
    print(cdsViewName)
    tableName = csvData.at[0,"CDS View"] + "_" + csvData.at[0,"Source.Name"]
    baseTable = csvData.at[0,"Source.Name"]
    csvData = csvData.replace({'Type':typeDict})

    # get primary key fields
    csvData["columnList"] = csvData["Technical Field Name(ODS)"] + " " + csvData["Type"] + ","
    dfKeys = csvData.query('PK=="yes"')["Technical Field Name(ODS)"]
    unique_fields=[]
    unique_fields = dfKeys.values.tolist()
    unique_fields_str = ','.join(unique_fields)
   
    # get increment field
    incField = csvData.query('`Incremental Field`=="yes"')["Technical Field Name(ODS)"]
    print(incField)
    incremental_fields=[]
    incremental_fields = incField.values.tolist()
    #title forces proper case
    incremental_fields_str = ','.join(incremental_fields)#.title()

    if incremental_fields ==[]:
        SAPAPI_strategy = 'insert-overwrite'
    else:
        SAPAPI_strategy = 'incremental'
    #add section to config
    try:
        config.add_section(cdsViewName)
    except DuplicateSectionError as error:
        print(cdsViewName,'Config already created')

    #update config
    config[cdsViewName] = {
            'active':'true',
            'client' :SAPAPI_Client,
            'sap_user':SAPAPI_user,
            'cds_view_name':cdsViewName,
            'base_table_name':baseTable,
            'key_fields':unique_fields_str,
            'strategy':SAPAPI_strategy,
            'incremental_field':incremental_fields_str,
            'row_limit':'5000',
            'select':'',
            'filter':'',
            'top':'',
            'skip':'',
            'inlinecount':'',
            'orderby':''
        }
    #write config
    writeConfigParser(cdsViewConfigFile, config)
    

['C:\\Repo\\EnterpriseCloud\\Notebooks\\CSVTemplates\\zcdv_t_cmtu062-T001-full.csv', 'C:\\Repo\\EnterpriseCloud\\Notebooks\\CSVTemplates\\zcdv_t_cmtu087-ADRC-incremental.csv', 'C:\\Repo\\EnterpriseCloud\\Notebooks\\CSVTemplates\\zcdv_t_cmtu092-CSKT-incremental.csv', 'C:\\Repo\\EnterpriseCloud\\Notebooks\\CSVTemplates\\zcdv_t_cmtu096-KNMT-incremental.csv', 'C:\\Repo\\EnterpriseCloud\\Notebooks\\CSVTemplates\\zcdv_t_cmtu098-LFB1-incremental.csv', 'C:\\Repo\\EnterpriseCloud\\Notebooks\\CSVTemplates\\zcdv_t_cmtu099-LFBK-full.csv', 'C:\\Repo\\EnterpriseCloud\\Notebooks\\CSVTemplates\\zcdv_t_cmtu100-LFM1-incremental.csv', 'C:\\Repo\\EnterpriseCloud\\Notebooks\\CSVTemplates\\zcdv_t_cmtu109-MSKA-incremental.csv', 'C:\\Repo\\EnterpriseCloud\\Notebooks\\CSVTemplates\\zcdv_t_cmtu110-MSKU-incremental.csv', 'C:\\Repo\\EnterpriseCloud\\Notebooks\\CSVTemplates\\zcdv_t_cmtu111-MSLB-incremental.csv', 'C:\\Repo\\EnterpriseCloud\\Notebooks\\CSVTemplates\\zcdv_t_cmtu112-MSPR-incremental.csv', 'C:\\Repo\\E

In [30]:
# create the fileupoader.config entries 
# (remove existing entries if you need to overwrite them in teh fileuploader.config file)
#read existing file
configFileList = getFile(configFile)

# create new list, remove empty lines
newFileList = []
for l in configFileList: 
    if l != "":
        newFileList.append(l)
  
# remove last line
newFileList = newFileList[:-1]

# create a working copy of df_tables to build config
df_tables_config = df_tables.copy()

#check to see if entry exists in file uploader config, if it does drop it from the df
for idx, row in df_tables_config.iterrows():
    dfTableName = row['CDS View']  
    config_exists = False
    for tn in newFileList:
        if dfTableName in tn:
            config_exists = True
    if config_exists:
        print('CDS view already in file uploader config, dropping',dfTableName,idx)        
        df_tables_config = df_tables_config.drop(idx)
            
#create the new config entries
confstr = ""

for l in newFileList:
    confstr += l

confstr = confstr.rstrip()
confstr = confstr.rstrip('\r\n')
confstr = confstr.rstrip()
for idx, row in df_tables_config.iterrows():
    confTableName = row['TABLE_NAME'] 
    confFileName = row['CDS View']
    confstr += """,
    """ +'"'+ confFileName + """\": {
        "SNOWFLAKE_SCHEMA_NAME": \"""" + lzSchema + """\",
        "SNOWFLAKE_DATABASE_NAME": "<<LZ>>",
        "SNOWFLAKE_WAREHOUSE_NAME": "Import_File_WH",
        "SNOWFLAKE_ROLE_NAME": "<<SYSADMIN>>",
        "SNOWFLAKE_TABLE_NAME": \"""" + confTableName + """\",
        "SNOWFLAKE_FILE_FORMAT_NAME": "CONTROL.POWERSHELL_JSON",
        "SNOWFLAKE_STAGE_TYPE": "NAMED",
        "SNOWFLAKE_STAGE_NAME": "SAPAPI_STG",
        "FILE": "F:\\\\Shares\\\\Files\\\\SAPH-API\\\\""" + confFileName + """.*.json",
        "SPLIT_FILE": false
    }"""
    
# add final bracket
confstr += """
}"""

# write out new config file
#print(confstr)
writeFile(configFile,confstr)

In [31]:
#**************************** LZ to ODS dbt model Utility**********************
# generate helper functions for DBT model

def getJinjaConfig(strategy,cdsView,unique_keys):

	jinjaConfig = """
	{{ config(
	  materialized='incremental',
	  incremental_strategy='"""+ strategy +"""',
	  alias='""" + cdsView + """',
	  unique_key = ['"""+ unique_keys +"""'],
	  transient=false
	) }} """

	return jinjaConfig

#CREATE DBT MODEL
#(inputs are column list for source_cte,transform_cte and tablename )
def getDbtCode(col_s_string,col_t_string,col_type_data_str,tablename,odsSchema,jconfig,ifMerge):

	DBT_CODE = """

	/************
	Model file generated from SAPAPIUtility 
	*************/

	"""+ jconfig +"""

	with source_data as (
	SELECT
		""" +col_s_string+ """,
		_FILEDATE,
		_FILENAME
	from """ + "{{ source('"+ odsSchema + "','" + tablename + "') }}" """
	"""+ ifMerge +"""

	),
	
	transform_data as (
	SELECT 
		""" +col_t_string+ """,
		_FILEDATE,
		_FILENAME,
		'{{ invocation_id }}' AS AUDIT_LOG_ID,
        SHA2('hash_me') as _ROW_HASH -- Adds column to table on full refresh.
	from source_data
	),
	
	type_data as (
	SELECT
		"""+col_type_data_str+""",
		cast(_FILEDATE AS TIMESTAMP_NTZ(9)) AS _FILEDATE,
		cast(_FILENAME as varchar(128)) as _FILENAME,
        cast(AUDIT_LOG_ID as varchar(128)) as AUDIT_LOG_ID,
        cast(_ROW_HASH as varchar(128)) as _ROW_HASH
	from transform_data
	)

	select * from type_data

			"""
	#print(DBT_CODE)
	return DBT_CODE
#DBT CODE FOR INSERT-OVERWRITES
# def getDbtCode_insertoverwrite(col_s_string,col_t_string,col_type_data_str,tablename,odsSchema,jconfig):

# 	DBT_CODE_I = """

# 	/************
# 	Model file generated from gmxDbtTemplateUtility.ipynb 
# 	*************/

# 	"""+ jconfig +"""

#     with max_f as (
# 		SELECT MAX(_FILENAME) as _filename 
# 		from """ + "{{ source('"+ odsSchema + "','" + tablename + "') }}"  """
# 	),

#     max_d as (
# 		select s._filename, max(s._filedate) as _filedate from """ + "{{ source('"+ odsSchema + "','" + tablename + "') }}"  """ as s
# 		join max_f on s._filename = max_f._filename
# 		group by s._filename
# 	),

# 	source_data as (
# 	SELECT
# 		""" +col_s_string+ """,
# 		s._FILEDATE,
# 		s._FILENAME
# 	from """ + "{{ source('"+ odsSchema + "','" + tablename + "') }} s"  """
#     join max_d on s._filedate = max_d._filedate and s._filename = max_d._filename
	
# 	),
	
# 	transform_data as (
# 	SELECT 
# 		""" +col_t_string+ """,
# 		_FILEDATE,
# 		_FILENAME,
# 		'{{ invocation_id }}' AS AUDIT_LOG_ID,
#         SHA2('hash_me') as _ROW_HASH -- Adds column to table on full refresh.
# 	from source_data
# 	),
	
# 	type_data as (
# 	SELECT
# 		"""+col_type_data_str+""",
# 		cast(_FILEDATE AS TIMESTAMP_NTZ(9)) AS _FILEDATE,
# 		cast(_FILENAME as varchar(128)) as _FILENAME,
#         cast(AUDIT_LOG_ID as varchar(128)) as AUDIT_LOG_ID,
#         cast(_ROW_HASH as varchar(128)) as _ROW_HASH
# 	from transform_data
# 	)

# 	select * from type_data

# 			"""
# 	#print(DBT_CODE)
# 	return DBT_CODE_I
#CREATE SCHEMA FILE
def createSchema(df_tables_test,lzSchema,odsSchema,schemaPath):
        df_tables_test['schema'] = """\n      - name: """ + df_tables_test['TABLE_NAME'] + """\n        identifier: """ + df_tables_test['TABLE_NAME'] + """\n        description:  Landing table for """ + df_tables_test['TABLE_NAME'] + """ file. """

        schema_str = """version: 2\nsources:\n  - name: """ + odsSchema + """\n    description: Landing Zone - """ + lzSchema + """ Schema\n    database: "{{ env_var('SNOWFLAKE_LZ') }}"\n    schema: """+ lzSchema + """\n    tables:"""

        for idx, r in df_tables_test.iterrows():
            schema_str += r['schema'] + """
        """
        df_tables_test['models'] = """\n  - name: """ + df_tables_test['TABLE_NAME'] + """\n    identifier: """ + df_tables_test['TABLE_NAME'] + """\n    description:  Landing table for """ + df_tables_test['TABLE_NAME'] + """ file. """

        schema_str_t ="""\nmodels:"""
        for idx, r in df_tables_test.iterrows():
            schema_str_t += r['models'] + """
        """
        final_schema = schema_str + schema_str_t
        	
        path_create = schemaPath
        filename = "schema.yml"
        path_write = path_create +"\\"+ filename

        # createFile(path_create,filename)
        writeFile(path_write,final_schema)
		

In [32]:
#append Schema.yml file
dbt_path = repo_path + "dbt\\lz_to_ods\\models\\sapapi\\"
df_tables_test = pd.DataFrame(df_tables["TABLE_NAME"])


schema_path = dbt_path + "schema.yml"
existing_schemas = []
#if schema file exists
#look for schema names
#add to list
if os.path.exists(schema_path):
    schema_file = getFile(schema_path)
    for line in schema_file:        
        if 'identifier:' in line:
            cdsName = line.replace("identifier:","")
            cdsName = cdsName.replace("'","")
            cdsName = cdsName.replace(",","")
            cdsName = cdsName.strip()
            existing_schemas.append(cdsName)
#remove any duplicates
existing_schemas = list(dict.fromkeys(existing_schemas))
#convert to dataframe
es = pd.DataFrame({'TABLE_NAME':existing_schemas})
#merge lists
df_tables_test = pd.concat([df_tables_test,es],ignore_index=True)
df_tables_test = df_tables_test.drop_duplicates()
createSchema(df_tables_test,lzSchema,odsSchema,dbt_path)

In [33]:
# Build dbt models from cds view templates

# read csv template file and build list of required inputs
for csvFile in filesList:
    
    dashChar = csvFile.rfind("-")
    dotChar = csvFile.rfind(".")
    dbt_strategy = csvFile[dashChar+1:dotChar]
    columnList = ""
    csvDataOut = pd.DataFrame()
    csvData = pd.read_csv(csvFile)
    #loop through characters df and replace any illegal characters
    for idx,row in df_characters.iterrows():
        csvData["ScrubbedColumnName"]  = csvData["Technical Field Name(ODS)"].str.replace(row['OLD'],row['NEW'])
    csvData["ScrubbedColumnName"] = csvData["ScrubbedColumnName"].str.upper()

    cdsViewName = csvData.at[0,"CDS View"]
    tableName = csvData.at[0,"CDS View"] + "_" + csvData.at[0,"Source.Name"]
    baseTable = csvData.at[0,"Source.Name"]
    csvData = csvData.replace({'Type':typeDict})
    csvData["columnList"] = csvData["ScrubbedColumnName"] + " " + csvData["Type"] + ","
    #print(csvData.head(5))
    
    #empty list for transform_cte,source_cte,type_data_cte columns 
    col_t = []
    col_s = []
    col_source = []
    col_type_data = []
    
    
    # build column strings
    for idx,row in csvData.iterrows():
        col_type_data.append('CAST("' + row["ScrubbedColumnName"] + '" AS ' + row["Type"] + ') AS "' + row["ScrubbedColumnName"] + '"')
        col_t.append(row["ScrubbedColumnName"])
        col_s.append('get_ignore_case(to_object(JSONFIELD),\'' + row["Technical Field Name(ODS)"] + '\')::STRING AS "' + row["ScrubbedColumnName"] + '"')  

    col_type_data_str = ', \n \t \t'.join(col_type_data)
    #col_type_data_str = col_type_data_str.upper()
    #print(col_type_data_str)

    # column string for transform cte
    col_t_string = '"' + '", \n \t \t"'.join(col_t) + '"'
    #col_t_string = col_t_string.casefold()
    
    col_s_string = ', \n \t \t'.join(col_s)
    #print(col_s_string)
        
    # build key structure
    dfKeys = csvData.query('PK=="yes"')["Technical Field Name(ODS)"]
    unique_fields=[]
    unique_fields = dfKeys.values.tolist()
    unique_fields_str = "','".join(unique_fields)
    unique_fields_qualify = []
    dbt_strategy = dbt_strategy.strip()
    if dbt_strategy == 'incremental':
        dbt_strategy = 'merge'
        
        unique_fields_qualify = ["get_ignore_case(to_object(JSONFIELD),'{0}')::STRING".format(k) for k in unique_fields]
        unique_fields_qualify_str = ",".join(unique_fields_qualify)
        
        ifMerge = """{% if is_incremental() %}WHERE  _FILEDATE > IFNULL((SELECT MAX(_FILEDATE) FROM {{ this }}),'1970-01-01'){% endif %}
        QUALIFY (ROW_NUMBER() OVER (PARTITION BY """+unique_fields_qualify_str+"""  ORDER BY _FILENAME DESC, _FILEDATE DESC )) = 1"""  

    else:
        dbt_strategy = 'insert_overwrite'
        ifMerge = """where REGEXP_REPLACE(split_part(_filename, '.',  2),'-[^-]*$','') = (SELECT MAX(REGEXP_REPLACE(split_part(_filename, '.',  2),'-[^-]*$','')) from """ + "{{ source('"+ odsSchema + "','" + tableName + "') }})" """"""
        unique_fields_str = ''
    
    #create dbt model jinja config
    jinjaConfig = getJinjaConfig(dbt_strategy,tableName,unique_fields_str)
    dbt_model = getDbtCode(col_s_string,col_t_string,col_type_data_str,tableName,odsSchema,jinjaConfig,ifMerge)
    # if dbt_strategy == 'merge':
    #     dbt_model = getDbtCode(col_s_string,col_t_string,col_type_data_str,tableName,odsSchema,jinjaConfig,ifMerge)
    # else:
    #     dbt_model = getDbtCode_insertoverwrite(col_s_string,col_t_string,col_type_data_str,tableName,odsSchema,jinjaConfig)
    #Creating code for dbt model
    #dbt_model = getDbtCode(col_s_string,col_t_string,col_type_data_str,tableName,odsSchema,jinjaConfig,ifMerge)
    # print(dbt_model)

    
    #create dbt file
    dbt_path = repo_path + "dbt\\lz_to_ods\\models\\sapapi\\"
    filename = tableName + ".sql"
    path_write = dbt_path + filename

    writeFile(path_write,dbt_model)
    print('created file',filename)
    

created file ZCDV_T_CMTU062_T001.sql
created file ZCDV_T_CMTU087_ADRC.sql
created file ZCDV_T_CMTU092_CSKT.sql
created file ZCDV_T_CMTU096_KNMT.sql
created file ZCDV_T_CMTU098_LFB1.sql
created file ZCDV_T_CMTU099_LFBK.sql
created file ZCDV_T_CMTU100_LFM1.sql
created file ZCDV_T_CMTU109_MSKA.sql
created file ZCDV_T_CMTU110_MSKU.sql
created file ZCDV_T_CMTU111_MSLB.sql
created file ZCDV_T_CMTU112_MSPR.sql
created file ZCDV_T_CMTU115_TKA01.sql
created file ZCDV_T_CMTU128_T161.sql
created file ZCDV_T_CMTU129_T161T.sql
created file ZCDV_T_CMTU130_T163I.sql
created file ZCDV_T_CMTU131_T163K.sql
created file ZCDV_T_CMTU132_T163Y.sql
created file ZCDV_T_CMTU133_T171.sql
created file ZCDV_T_CMTU134_T176T.sql
created file ZCDV_T_CMTU135_T178T.sql
created file ZCDV_T_CMTU136_T179T.sql
created file ZCDV_T_CMTU137_T683T.sql
created file ZCDV_T_CMTU138_T685.sql
created file ZCDV_T_CMTU139_T685T.sql
created file ZCDV_T_CMTU140_T691T.sql
created file ZCDV_T_CMTU141_T880.sql
created file ZCDV_T_CMTU142

Append ODS_TO_PUB schema


In [62]:
def createSchema_pub(df_tables_test,schemaPath):
        df_tables_test['schema'] = """\n      - name: """ + df_tables_test['TABLE_NAME'] + """\n        identifier: """ + df_tables_test['TABLE_NAME'] + """\n        description:  Landing table for """ + df_tables_test['TABLE_NAME'] +""""""

        schema_str = """version: 2\nsources:\n  - name: ODS_SAPAPI\n    description: Landing Zone - SAPAPI Schema\n    database: "{{ env_var('SNOWFLAKE_ODS') }}"\n    schema: SAPAPI\n    tables:"""

        for idx, r in df_tables_test.iterrows():
            schema_str += r['schema'] + """
        """
        df_tables_test['models'] = """\n  - name: """ + df_tables_test['TABLE_NAME'] + """\n    identifier: """ + df_tables_test['TABLE_NAME'] + """\n    description:  Landing table for """ + df_tables_test['TABLE_NAME'] + """"""

        # schema_str_t ="""\nmodels:"""
        
        # for idx, r in df_tables_test.iterrows():
        #     schema_str_t += r['models'] + """
        # """
        final_schema = schema_str
        	
        path_create = schemaPath
        filename = "schema.yml"
        path_write = path_create +"\\"+ filename

        # createFile(path_create,filename)
        writeFile(path_write,final_schema)

In [63]:
dbt_path = repo_path + "dbt\\ods_to_pub\\models\\sapapi\\"
df_tables_test = pd.DataFrame(df_tables["TABLE_NAME"])


schema_path = dbt_path + "schema.yml"
existing_schemas = []
#if schema file exists
#look for schema names
#add to list
if os.path.exists(schema_path):
    schema_file = getFile(schema_path)
    for line in schema_file:        
        if 'identifier:' in line:
            cdsName = line.replace("identifier:","")
            cdsName = cdsName.replace("'","")
            cdsName = cdsName.replace(",","")
            cdsName = cdsName.strip()
            existing_schemas.append(cdsName)
#remove any duplicates
existing_schemas = list(dict.fromkeys(existing_schemas))
#convert to dataframe
es = pd.DataFrame({'TABLE_NAME':existing_schemas})
#merge lists
df_tables_test = pd.concat([df_tables_test,es],ignore_index=True)
createSchema_pub(df_tables_test,dbt_path)

In [64]:
## create pub ODS_ views
%run ..\\Python\\dbtTools\\dbtGeneratePubViews.py "ODS" "SAPAPI" "C:\Repo\EnterpriseCloud\dbt\ods_to_pub\models" "varun.x.tiwari@us.tel.com"

Arguments passed: DB: ODS  Schema: SAPAPI Path: C:\Repo\EnterpriseCloud\dbt\ods_to_pub\models user: varun.x.tiwari@us.tel.com
Initiating login request with your identity provider. A browser window should have opened for you to complete the login. If you can't see it, check existing browser windows, or your OS settings. Press CTRL+C to abort and try again...
Schema.yml already exists, delete to refresh.
ODS ODS {{ source('ODS_SAPAPI','ZCDV_T_CMTU009_VBAK') }}
C:\Repo\EnterpriseCloud\dbt\ods_to_pub\models\sapapi\v_ZCDV_T_CMTU009_VBAK.sql already exists, delete to refresh.
ODS ODS {{ source('ODS_SAPAPI','ZCDV_T_CMTU014_VBRK') }}
C:\Repo\EnterpriseCloud\dbt\ods_to_pub\models\sapapi\v_ZCDV_T_CMTU014_VBRK.sql already exists, delete to refresh.
ODS ODS {{ source('ODS_SAPAPI','ZCDV_T_CMTU015_VBRP') }}
C:\Repo\EnterpriseCloud\dbt\ods_to_pub\models\sapapi\v_ZCDV_T_CMTU015_VBRP.sql already exists, delete to refresh.
ODS ODS {{ source('ODS_SAPAPI','ZCDV_T_CMTU017_MARA') }}
C:\Repo\EnterpriseCloud\